# MedScribe — Person 3: SOAP note generation (Colab)

**What it does:** for each of the three ACI-Bench dialogues it produces 2 notes — **rollu** (`gold_role`) and **rolsuz** (`None`) — 6 notes in total (plan section 3).

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Time:** first run ~4 min for the model download + ~4 min for the 6 notes.

**Input:** the GitHub repository (cloned below). The prompt is **frozen** in `notes/istem_v1.txt`; there is no copy of it in code.

**Output:** `notes/outputs/aci_00X_rollu.json`, `..._rolsuz.json`, `comparison.md`. These are committed to the repository (Tuesday deliverable).

In [ ]:
# 1) Hardware
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: select Runtime -> T4 GPU'

In [ ]:
# 2) Dependencies (~1 min)
!pip -q install -U transformers accelerate bitsandbytes

In [ ]:
# 3) Get the repository
REPO = "https://github.com/Pikseel/medscribe.git"
PROJECT = "/content/medscribe"

import os, sys
if not os.path.isdir(PROJECT):
    !git clone -q $REPO $PROJECT
%cd $PROJECT
!git pull -q
sys.path.insert(0, PROJECT)
print(sorted(os.listdir('.')))

In [ ]:
# 4) Validate the data format (common/format.py)
!python -m common.format notes/data/aci_001.json notes/data/aci_002.json notes/data/aci_003.json

In [ ]:
# 5) Load the model: 4-bit Qwen2.5-7B-Instruct (fits on a T4)
import notes.generate_note as gn
gn.set_backend('transformers')
gn._load_hf()
print('model ready:', gn.model_name(), '| backend:', gn.BACKEND)

In [ ]:
# 6) Prompt check: the plan section 2.5b example dialogue
import json
d = json.load(open('common/ornek_diyalog.json'))
note = gn.generate_note(d, 'gold_role')
print(json.dumps(note, ensure_ascii=False, indent=2))
# Expected: cough for a week, mild fever at night, NO chest pain.
# If 'chest pain' is written as present, the prompt is too weak -> istem_v2 is needed, STOP.

In [ ]:
# 7) Experiment: 3 dialogues x 2 conditions = 6 notes
import time
rows = []
for path, rf, cond in gn.EXPERIMENT:
    d = json.load(open(path))
    t0 = time.time()
    note = gn.generate_note(d, rf)
    out = f"notes/outputs/{d['dialogue_id']}_{cond}.json"
    gn.save_note(note, out)
    rows.append((d['dialogue_id'], cond, len(note['S']), len(note['O']),
                 len(note['A']), len(note['P']), round(time.time() - t0)))
    print(f"{out:44s} {rows[-1][6]:3d}s")

print()
print(f"{'dialogue':10s} {'condition':9s} {'S':>5s} {'O':>5s} {'A':>5s} {'P':>5s}")
for r in rows:
    print(f"{r[0]:10s} {r[1]:9s} {r[2]:5d} {r[3]:5d} {r[4]:5d} {r[5]:5d}")

In [ ]:
# 8) Rollu vs rolsuz comparison + negation flags
!python -m notes.compare_notes
from IPython.display import Markdown, display
display(Markdown(open('notes/outputs/comparison.md', encoding='utf-8').read()))

In [ ]:
# 9) Download the results (+ optionally copy to Drive)
!cd /content && zip -q -r kisi3_notes.zip medscribe/notes/outputs
from google.colab import files
files.download('/content/kisi3_notes.zip')

SAVE_TO_DRIVE = False   # set to True to copy into MedScribe_Drive/notes_outputs
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/MedScribe_Drive/notes_outputs
    !cp notes/outputs/*.json notes/outputs/comparison.md /content/drive/MyDrive/MedScribe_Drive/notes_outputs/